# EDA 4 — Boundary Effects: London-Only vs National Frame

**Purpose:** Quantify how including London-external flows (via the +1 synthetic MSOA) changes the cascade-counter balance, and whether this alters the typology or its validation.

### Three questions
1. **Where do external flows land?** How external volumes distribute across the decile hierarchy, and why ~69% of London MSOAs receive them as "wealthier" connections.
2. **What shifts in the cascade-counter balance?** Per-MSOA and per-decile dominance shifts between London-only and national frames.
3. **Does the typology change?** Compute missing national-frame derived metrics, re-classify, and compare.

### Data source
`msoa_cascade_national_frame_20260623.csv` — 983 London MSOAs with both London-only (`_11`/`_21`) and national-frame (`_nat_11`/`_nat_21`) cascade metrics.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight'
})

# ── Paths (adjust to match your setup) ────────────────────────
# If using pyprojroot:
# from pyprojroot import here
# ROOT = here()
# DATA_DIR = ROOT / 'outputs'
# OUTPUT_DIR = ROOT / 'outputs/eda_figs'
# GEO_PATH = ROOT / 'data/london_msoa_2011.geojson'

# Fallback for running standalone:
DATA_DIR = Path('.')        # adjust
OUTPUT_DIR = Path('.')      # adjust
# GEO_PATH = Path('data/london_msoa_2011.geojson')  # uncomment if available

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Load data ─────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'msoa_cascade_national_frame_20260623.csv')
print(f'Loaded: {df.shape[0]} MSOAs, {df.shape[1]} columns')

# Quick check: national columns present
nat_cols = [c for c in df.columns if '_nat_' in c]
print(f'National-frame columns: {len(nat_cols)}')

---
## 1. Compute Missing National-Frame Derived Metrics

The preprocessing exported base metrics and `Cascade_Dominance_nat`, but not:
- `Cross_Decile_Share_nat` — share of total migration crossing decile boundaries
- `Sign_Concordance_nat` — whether Net_Cascade and Net_Counter share the same sign

We compute these here using the same formulas as EDA 1.

In [ ]:
# ── Cross_Decile_Share_nat ────────────────────────────────────
# Formula: (CFI_Churn + Counter_Churn) / Total_Migration
for yr in ['11', '21']:
    cfi  = df[f'CFI_Churn_nat_{yr}']
    cc   = df[f'Counter_Churn_nat_{yr}']
    tmig = df[f'Total_Migration_nat_{yr}']
    df[f'Cross_Decile_Share_nat_{yr}'] = np.where(
        tmig > 0, (cfi + cc) / tmig, 0)

# ── Sign_Concordance_nat ──────────────────────────────────────
for yr in ['11', '21']:
    nc  = df[f'Net_Cascade_nat_{yr}']
    ncc = df[f'Net_Counter_nat_{yr}']
    df[f'Sign_Concordance_nat_{yr}'] = np.where(
        (nc == 0) | (ncc == 0), 'zero',
        np.where(np.sign(nc) == np.sign(ncc), 'concordant', 'divergent')
    )

# ── Verify ────────────────────────────────────────────────────
print('=== Cross_Decile_Share_nat (2021) ===')
print(df['Cross_Decile_Share_nat_21'].describe().round(4).to_string())
print(f'\n=== Sign_Concordance_nat (2021) ===')
print(df['Sign_Concordance_nat_21'].value_counts().to_string())

---
# PART 1: Where Do External Flows Land?

External flows connect London MSOAs to the synthetic `EXT_OUTSIDE` MSOA (assigned national D6). Because ~69% of London MSOAs sit in national D1–D5, for those areas external inflows are `Inflow_Wealthier` and external outflows are `Outflow_Wealthier`.

This section quantifies:
- Aggregate external flow volumes (2011 vs 2021)
- How external inflows and outflows distribute across the decile hierarchy
- Which MSOAs receive the largest external flow injections

---

## 1a. Aggregate External Flow Summary

In [ ]:
print('=' * 65)
print('AGGREGATE EXTERNAL FLOW SUMMARY')
print('=' * 65)

for yr in ['11', '21']:
    ei = df[f'Ext_Inflow_nat_{yr}']
    eo = df[f'Ext_Outflow_nat_{yr}']
    en = df[f'Ext_Net_nat_{yr}']
    
    print(f'\n── 20{yr} ──')
    print(f'  External inflows  (Ext→Ldn):  {ei.sum():>10,.0f} total, '
          f'{ei.mean():>6.1f} per MSOA')
    print(f'  External outflows (Ldn→Ext):  {eo.sum():>10,.0f} total, '
          f'{eo.mean():>6.1f} per MSOA')
    print(f'  Net external:                 {en.sum():>10,.0f} total, '
          f'{en.mean():>+6.1f} per MSOA')
    print(f'  London is a net {"EXPORTER" if en.sum() < 0 else "IMPORTER"} '
          f'of {abs(en.sum()):,.0f} migrants')
    print(f'  MSOAs with zero external flows: {(ei == 0).sum()}')

# Temporal change
ei_11, ei_21 = df['Ext_Inflow_nat_11'].sum(), df['Ext_Inflow_nat_21'].sum()
eo_11, eo_21 = df['Ext_Outflow_nat_11'].sum(), df['Ext_Outflow_nat_21'].sum()
print(f'\n── Temporal Change ──')
print(f'  Inflow change:  {ei_21 - ei_11:>+10,.0f}  ({(ei_21/ei_11 - 1)*100:>+.1f}%)')
print(f'  Outflow change: {eo_21 - eo_11:>+10,.0f}  ({(eo_21/eo_11 - 1)*100:>+.1f}%)')
print(f'  ⟹ External outflow nearly {eo_21/eo_11:.1f}x the 2011 level')

## 1b. How External Flows Are Classified by Decile

The synthetic external MSOA is assigned national D6. For each London MSOA, whether the external flow counts as "wealthier" or "poorer" depends on that MSOA's own national decile. This table shows the classification logic directly.

In [ ]:
EXT_DECILE = 6  # synthetic external MSOA assigned D6

print('=' * 72)
print('HOW EXTERNAL FLOWS ARE CLASSIFIED BY NATIONAL DECILE')
print('=' * 72)
print(f'\n  External MSOA decile: D{EXT_DECILE}')
print(f'  Convention: D1 = most deprived, D10 = least deprived\n')
print(f'{"Nat Decile":>10s} {"N MSOAs":>8s} {"Ext inflow is":>18s} {"Ext outflow is":>18s}')
print('-' * 58)

for d in range(1, 11):
    n = (df['Wealth_Decile_National'] == d).sum()
    if d < EXT_DECILE:
        inflow_type  = 'Inflow_Wealthier'
        outflow_type = 'Outflow_Wealthier'
    elif d > EXT_DECILE:
        inflow_type  = 'Inflow_Poorer'
        outflow_type = 'Outflow_Poorer'
    else:
        inflow_type  = 'LATERAL (same)'
        outflow_type = 'LATERAL (same)'
    print(f'  D{d:>2d}     {n:>5d}     {inflow_type:<18s} {outflow_type:<18s}')

below = (df['Wealth_Decile_National'] < EXT_DECILE).sum()
same  = (df['Wealth_Decile_National'] == EXT_DECILE).sum()
above = (df['Wealth_Decile_National'] > EXT_DECILE).sum()
print(f'\n  Summary:')
print(f'    D1–D5 (below D6): {below} MSOAs ({below/len(df)*100:.1f}%) '
      f'→ external = WEALTHIER connection')
print(f'    D6 (same):        {same} MSOAs ({same/len(df)*100:.1f}%) '
      f'→ external = LATERAL (no cascade effect)')
print(f'    D7–D10 (above):   {above} MSOAs ({above/len(df)*100:.1f}%) '
      f'→ external = POORER connection')

## 1c. External Flow Volumes by National Decile

In [ ]:
# ── Table: external inflow & outflow by decile ────────────────
print('=' * 80)
print('EXTERNAL FLOW VOLUMES BY NATIONAL DECILE')
print('=' * 80)

dec_col = 'Wealth_Decile_National'
rows = []
for d in range(1, 11):
    mask = df[dec_col] == d
    row = {'Decile': d, 'N': mask.sum()}
    for yr in ['11', '21']:
        row[f'Ext_In_{yr}']  = df.loc[mask, f'Ext_Inflow_nat_{yr}'].sum()
        row[f'Ext_Out_{yr}'] = df.loc[mask, f'Ext_Outflow_nat_{yr}'].sum()
        row[f'Ext_Net_{yr}'] = df.loc[mask, f'Ext_Net_nat_{yr}'].sum()
    rows.append(row)

ext_dec = pd.DataFrame(rows)

print(f'\n{"Decile":>6s} {"N":>5s}  '
      f'{"Ext_In_11":>10s} {"Ext_Out_11":>10s} {"Ext_Net_11":>10s}  '
      f'{"Ext_In_21":>10s} {"Ext_Out_21":>10s} {"Ext_Net_21":>10s}')
print('-' * 80)
for _, r in ext_dec.iterrows():
    print(f'  D{r["Decile"]:>2.0f}  {r["N"]:>4.0f}  '
          f'{r["Ext_In_11"]:>10,.0f} {r["Ext_Out_11"]:>10,.0f} {r["Ext_Net_11"]:>+10,.0f}  '
          f'{r["Ext_In_21"]:>10,.0f} {r["Ext_Out_21"]:>10,.0f} {r["Ext_Net_21"]:>+10,.0f}')
print('-' * 80)
print(f'  Total {len(df):>4d}  '
      f'{ext_dec["Ext_In_11"].sum():>10,.0f} {ext_dec["Ext_Out_11"].sum():>10,.0f} '
      f'{ext_dec["Ext_Net_11"].sum():>+10,.0f}  '
      f'{ext_dec["Ext_In_21"].sum():>10,.0f} {ext_dec["Ext_Out_21"].sum():>10,.0f} '
      f'{ext_dec["Ext_Net_21"].sum():>+10,.0f}')

In [ ]:
# ── Fig 16: External flow volumes by decile ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x = np.arange(1, 11)
w = 0.35

# Panel (a): External inflow by decile
ax = axes[0]
ax.bar(x - w/2, ext_dec['Ext_In_11'], w, label='2011', color='#2166ac', alpha=0.7)
ax.bar(x + w/2, ext_dec['Ext_In_21'], w, label='2021', color='#b2182b', alpha=0.7)
ax.set_xlabel('National Wealth Decile')
ax.set_ylabel('Total external inflow')
ax.set_title('(a) External Inflow by Decile')
ax.set_xticks(x)
ax.set_xticklabels([f'D{d}' for d in x])
ax.axvline(5.5, color='grey', ls=':', lw=1, alpha=0.5)
ax.text(3, ax.get_ylim()[1]*0.9, '← Ext = Wealthier', ha='center', fontsize=8, color='grey')
ax.text(8, ax.get_ylim()[1]*0.9, 'Ext = Poorer →', ha='center', fontsize=8, color='grey')
ax.legend(fontsize=9)

# Panel (b): External outflow by decile
ax = axes[1]
ax.bar(x - w/2, ext_dec['Ext_Out_11'], w, label='2011', color='#2166ac', alpha=0.7)
ax.bar(x + w/2, ext_dec['Ext_Out_21'], w, label='2021', color='#b2182b', alpha=0.7)
ax.set_xlabel('National Wealth Decile')
ax.set_ylabel('Total external outflow')
ax.set_title('(b) External Outflow by Decile')
ax.set_xticks(x)
ax.set_xticklabels([f'D{d}' for d in x])
ax.axvline(5.5, color='grey', ls=':', lw=1, alpha=0.5)
ax.text(3, ax.get_ylim()[1]*0.9, '← Ext = Wealthier', ha='center', fontsize=8, color='grey')
ax.text(8, ax.get_ylim()[1]*0.9, 'Ext = Poorer →', ha='center', fontsize=8, color='grey')
ax.legend(fontsize=9)

# Panel (c): Net external by decile
ax = axes[2]
ax.bar(x - w/2, ext_dec['Ext_Net_11'], w, label='2011', color='#2166ac', alpha=0.7)
ax.bar(x + w/2, ext_dec['Ext_Net_21'], w, label='2021', color='#b2182b', alpha=0.7)
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('National Wealth Decile')
ax.set_ylabel('Net external flow (In − Out)')
ax.set_title('(c) Net External Flow by Decile')
ax.set_xticks(x)
ax.set_xticklabels([f'D{d}' for d in x])
ax.legend(fontsize=9)

fig.suptitle('Fig 16: External Flow Volumes by National Wealth Decile', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_16_external_flows_by_decile.png', dpi=150)
plt.show()

## 1d. Per-MSOA External Flow Intensity

How large are external flows relative to total migration? This determines how much leverage boundary effects have on each MSOA's cascade metrics.

In [ ]:
# ── External share of total migration ─────────────────────────
for yr in ['11', '21']:
    ext_total = df[f'Ext_Inflow_nat_{yr}'] + df[f'Ext_Outflow_nat_{yr}']
    tmig = df[f'Total_Migration_nat_{yr}']
    df[f'Ext_Share_{yr}'] = np.where(tmig > 0, ext_total / tmig * 100, 0)

print('=== External flows as % of total migration ===')
for yr in ['11', '21']:
    col = f'Ext_Share_{yr}'
    print(f'\n  20{yr}:')
    print(f'    Mean:   {df[col].mean():.1f}%')
    print(f'    Median: {df[col].median():.1f}%')
    print(f'    P90:    {df[col].quantile(0.90):.1f}%')
    print(f'    Max:    {df[col].max():.1f}%')

In [ ]:
# ── Fig 17: External share by decile (box + strip) ────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
dec_col = 'Wealth_Decile_National'

for idx, yr in enumerate(['11', '21']):
    ax = axes[idx]
    col = f'Ext_Share_{yr}'
    
    data_by_dec = [df.loc[df[dec_col] == d, col].values for d in range(1, 11)]
    bp = ax.boxplot(data_by_dec, positions=range(1, 11), patch_artist=True,
                    widths=0.6, showfliers=False)
    for box in bp['boxes']:
        box.set_facecolor('#2166ac' if yr == '11' else '#b2182b')
        box.set_alpha(0.3)
    for med in bp['medians']:
        med.set_color('black')
        med.set_linewidth(2)
    
    # Overlay strip
    for d in range(1, 11):
        vals = df.loc[df[dec_col] == d, col].values
        jitter = np.random.normal(0, 0.08, size=len(vals))
        ax.scatter(d + jitter, vals, alpha=0.15, s=8,
                   color='#2166ac' if yr == '11' else '#b2182b')
    
    ax.set_xlabel('National Wealth Decile')
    ax.set_title(f'20{yr}')
    ax.set_xticks(range(1, 11))
    ax.set_xticklabels([f'D{d}' for d in range(1, 11)])

axes[0].set_ylabel('External flows as % of total migration')
fig.suptitle('Fig 17: External Flow Share by National Decile', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_17_external_share_by_decile.png', dpi=150)
plt.show()

## 1e. How External Flows Feed Cascade vs Counter Directions

For MSOAs below D6 (the majority), external inflow = `Inflow_Wealthier` (cascade component), and external outflow = `Outflow_Wealthier` (counter-cascade component). So external flows inject volume into *both* sides of the cascade-counter balance. The key question is: is the injection symmetric?

In [ ]:
# ── Decompose external flow contribution to cascade vs counter ──
print('=' * 72)
print('EXTERNAL FLOW CONTRIBUTION TO CASCADE vs COUNTER COMPONENTS')
print('=' * 72)

for yr in ['11', '21']:
    # Difference between national and London-only base flows
    iw_boost = df[f'Inflow_Wealthier_nat_{yr}'] - df[f'Inflow_Wealthier_{yr}']
    op_boost = df[f'Outflow_Poorer_nat_{yr}'] - df[f'Outflow_Poorer_{yr}']
    ow_boost = df[f'Outflow_Wealthier_nat_{yr}'] - df[f'Outflow_Wealthier_{yr}']
    ip_boost = df[f'Inflow_Poorer_nat_{yr}'] - df[f'Inflow_Poorer_{yr}']
    
    casc_boost  = iw_boost.sum() + op_boost.sum()   # CFI_Churn increase
    counter_boost = ow_boost.sum() + ip_boost.sum()  # Counter_Churn increase
    
    print(f'\n── 20{yr} ──')
    print(f'  Inflow_Wealthier boost:  {iw_boost.sum():>+10,.0f}  (cascade component)')
    print(f'  Outflow_Poorer boost:    {op_boost.sum():>+10,.0f}  (cascade component)')
    print(f'  Outflow_Wealthier boost: {ow_boost.sum():>+10,.0f}  (counter component)')
    print(f'  Inflow_Poorer boost:     {ip_boost.sum():>+10,.0f}  (counter component)')
    print(f'  ─────────────────────────────────')
    print(f'  Total cascade churn boost:       {casc_boost:>+10,.0f}')
    print(f'  Total counter-cascade churn boost:{counter_boost:>+10,.0f}')
    print(f'  Difference (cascade − counter):  {casc_boost - counter_boost:>+10,.0f}')
    print(f'  ⟹ External flows push {"CASCADE" if casc_boost > counter_boost else "COUNTER-CASCADE"} '
          f'harder by {abs(casc_boost - counter_boost):,.0f}')

In [ ]:
# ── Fig 18: Cascade vs counter churn boost by decile ──────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(1, 11)
w = 0.35

for idx, yr in enumerate(['11', '21']):
    ax = axes[idx]
    casc_by_dec = []
    counter_by_dec = []
    
    for d in range(1, 11):
        mask = df['Wealth_Decile_National'] == d
        iw_b = (df.loc[mask, f'Inflow_Wealthier_nat_{yr}'] - 
                df.loc[mask, f'Inflow_Wealthier_{yr}']).sum()
        op_b = (df.loc[mask, f'Outflow_Poorer_nat_{yr}'] - 
                df.loc[mask, f'Outflow_Poorer_{yr}']).sum()
        ow_b = (df.loc[mask, f'Outflow_Wealthier_nat_{yr}'] - 
                df.loc[mask, f'Outflow_Wealthier_{yr}']).sum()
        ip_b = (df.loc[mask, f'Inflow_Poorer_nat_{yr}'] - 
                df.loc[mask, f'Inflow_Poorer_{yr}']).sum()
        casc_by_dec.append(iw_b + op_b)
        counter_by_dec.append(ow_b + ip_b)
    
    ax.bar(x - w/2, casc_by_dec, w, label='Cascade churn boost', 
           color='#d6604d', alpha=0.7)
    ax.bar(x + w/2, counter_by_dec, w, label='Counter churn boost', 
           color='#4393c3', alpha=0.7)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xlabel('National Wealth Decile')
    ax.set_ylabel('Churn boost from external flows')
    ax.set_title(f'20{yr}')
    ax.set_xticks(x)
    ax.set_xticklabels([f'D{d}' for d in x])
    ax.legend(fontsize=9)

fig.suptitle('Fig 18: External Flow Contribution to Cascade vs Counter Churn by Decile', 
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_18_ext_cascade_counter_boost_by_decile.png', dpi=150)
plt.show()

---
# PART 2: What Shifts in the Cascade-Counter Balance?

---

## 2a. Headline Comparison: London-only vs National Frame

In [ ]:
print('=' * 80)
print('HEADLINE METRIC COMPARISON: London-only vs National Frame')
print('=' * 80)

metrics_compare = [
    ('CFI_Churn', 'Cascade churn'),
    ('Counter_Churn', 'Counter churn'),
    ('Net_Cascade', 'Net cascade'),
    ('Net_Counter', 'Net counter'),
    ('Cascade_Dominance', 'Cascade dominance'),
]

for yr in ['11', '21']:
    print(f'\n── 20{yr} {"─" * 60}')
    print(f'{"Metric":>22s} {"London mean":>12s} {"National mean":>14s} {"Difference":>12s}')
    print('-' * 65)
    for col, label in metrics_compare:
        ldn = df[f'{col}_{yr}'].mean()
        nat = df[f'{col}_nat_{yr}'].mean()
        diff = nat - ldn
        print(f'  {label:>20s}  {ldn:>11.2f}  {nat:>13.2f}  {diff:>+11.2f}')

In [ ]:
# ── Aggregate sums (not just means) ──────────────────────────
print('\n' + '=' * 80)
print('AGGREGATE SUMS (city-wide totals)')
print('=' * 80)

sum_metrics = ['Net_Cascade', 'Net_Counter', 'CFI_Churn', 'Counter_Churn']
for yr in ['11', '21']:
    print(f'\n── 20{yr} ──')
    for m in sum_metrics:
        ldn = df[f'{m}_{yr}'].sum()
        nat = df[f'{m}_nat_{yr}'].sum()
        print(f'  {m:>20s}:  London={ldn:>12,.0f}   National={nat:>12,.0f}   '
              f'Boost={nat-ldn:>+10,.0f}')

## 2b. Cascade Dominance Shift: London-only → National

In [ ]:
# ── Per-MSOA dominance shift ──────────────────────────────────
for yr in ['11', '21']:
    df[f'Dom_Shift_{yr}'] = (df[f'Cascade_Dominance_nat_{yr}'] - 
                              df[f'Cascade_Dominance_{yr}'])

print('=== Cascade Dominance Shift (national − London-only) ===')
for yr in ['11', '21']:
    col = f'Dom_Shift_{yr}'
    print(f'\n  20{yr}:')
    print(f'    Mean shift:   {df[col].mean():+.6f}')
    print(f'    Median shift: {df[col].median():+.6f}')
    print(f'    Std:          {df[col].std():.6f}')
    toward_casc    = (df[col] > 0.001).sum()
    toward_counter = (df[col] < -0.001).sum()
    negligible     = len(df) - toward_casc - toward_counter
    print(f'    Shifted toward cascade:        {toward_casc} ({toward_casc/len(df)*100:.1f}%)')
    print(f'    Shifted toward counter:        {toward_counter} ({toward_counter/len(df)*100:.1f}%)')
    print(f'    Negligible change (±0.001):    {negligible} ({negligible/len(df)*100:.1f}%)')

In [ ]:
# ── Fig 19: Dominance shift by national decile ────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
dec_col = 'Wealth_Decile_National'

for idx, yr in enumerate(['11', '21']):
    ax = axes[idx]
    col = f'Dom_Shift_{yr}'
    
    means = df.groupby(dec_col)[col].mean()
    sems  = df.groupby(dec_col)[col].sem()
    
    color = '#2166ac' if yr == '11' else '#b2182b'
    ax.bar(means.index, means.values, color=color, alpha=0.6, 
           yerr=sems.values, capsize=3)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xlabel('National Wealth Decile')
    ax.set_title(f'20{yr}')
    ax.set_xticks(range(1, 11))
    ax.set_xticklabels([f'D{d}' for d in range(1, 11)])
    
    # Annotate direction
    ax.text(0.02, 0.95, '↑ Toward cascade', transform=ax.transAxes,
            fontsize=8, color='grey', va='top')
    ax.text(0.02, 0.05, '↓ Toward counter', transform=ax.transAxes,
            fontsize=8, color='grey', va='bottom')

axes[0].set_ylabel('Cascade Dominance shift\n(national − London-only)')
fig.suptitle('Fig 19: Cascade Dominance Shift by National Decile', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_19_dominance_shift_by_decile.png', dpi=150)
plt.show()

In [ ]:
# ── Fig 20: Per-MSOA dominance shift distribution ─────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for idx, yr in enumerate(['11', '21']):
    ax = axes[idx]
    col = f'Dom_Shift_{yr}'
    color = '#2166ac' if yr == '11' else '#b2182b'
    
    ax.hist(df[col], bins=50, color=color, alpha=0.6, edgecolor='white', lw=0.3)
    ax.axvline(0, color='black', lw=1)
    ax.axvline(df[col].mean(), color=color, ls='--', lw=1.5,
               label=f'Mean: {df[col].mean():+.4f}')
    ax.axvline(df[col].median(), color=color, ls=':', lw=1.5,
               label=f'Median: {df[col].median():+.4f}')
    ax.set_xlabel('Dominance shift (national − London-only)')
    ax.set_title(f'20{yr}')
    ax.legend(fontsize=9)

axes[0].set_ylabel('Number of MSOAs')
fig.suptitle('Fig 20: Distribution of Per-MSOA Cascade Dominance Shift', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_20_dominance_shift_distribution.png', dpi=150)
plt.show()

## 2c. Cross_Decile_Share Comparison

In [ ]:
print('=== Cross_Decile_Share: London-only vs National ===')
for yr in ['11', '21']:
    ldn = df[f'Cross_Decile_Share_{yr}']
    nat = df[f'Cross_Decile_Share_nat_{yr}']
    print(f'\n  20{yr}:')
    print(f'    London mean:   {ldn.mean():.4f}  (median {ldn.median():.4f})')
    print(f'    National mean: {nat.mean():.4f}  (median {nat.median():.4f})')
    print(f'    Difference:    {nat.mean() - ldn.mean():+.4f}')
    print(f'    Correlation:   {ldn.corr(nat):.4f}')

## 2d. Sign Concordance Comparison

In [ ]:
print('=== Sign Concordance: London-only vs National ===')
for yr in ['11', '21']:
    ldn = df[f'Sign_Concordance_{yr}']
    nat = df[f'Sign_Concordance_nat_{yr}']
    print(f'\n── 20{yr} ──')
    print(f'  London-only:')
    for v in ['concordant', 'divergent', 'zero']:
        n = (ldn == v).sum()
        print(f'    {v:>12s}: {n:>4d} ({n/len(df)*100:.1f}%)')
    print(f'  National frame:')
    for v in ['concordant', 'divergent', 'zero']:
        n = (nat == v).sum()
        print(f'    {v:>12s}: {n:>4d} ({n/len(df)*100:.1f}%)')
    
    # Cross-tab
    print(f'\n  Cross-tab (rows=London, cols=National):')
    ct = pd.crosstab(ldn, nat)
    print(ct.to_string())
    changed = (ldn != nat).sum()
    print(f'  Changed classification: {changed} ({changed/len(df)*100:.1f}%)')

---
# PART 3: Does the Typology Change?

### Important methodological note

The national-frame metrics (`_nat_`) differ from London-only metrics in **two confounded ways**:
1. **Different decile frame**: London-only assigns deciles among 983 London MSOAs; national assigns among ~6,800 England MSOAs. Because London skews deprived nationally (~69% fall in national D1–D5), a London-only D5 might be national D3. Only 23.5% of MSOAs keep the same decile number in both frames.
2. **External flows added**: The +1 synthetic MSOA injects additional migration volume.

These two effects cannot be cleanly separated in the current data. A high typology switch rate therefore reflects **both** the re-framing of what counts as "cross-decile" **and** the additional external flows. This is by design — the national frame is a holistic sensitivity test — but it means typology switches should not be interpreted as purely an "external flow" effect.

---

## 3a. Typology Assignment: National Frame

Using the same thresholds as EDA 3:
- `Cascade_Dominance` > 0.52 → Cascade-led
- `Cascade_Dominance` < 0.48 → Counter-led
- Between 0.48 and 0.52 → Symmetric
- `Cross_Decile_Share` < P25 → Lateral

In [ ]:
def assign_typology(dom, cds, dom_upper=0.52, dom_lower=0.48, cds_threshold=None):
    """Classify MSOA into flow regime typology."""
    if cds < cds_threshold:
        return 'Lateral'
    elif dom > dom_upper:
        return 'Cascade-led'
    elif dom < dom_lower:
        return 'Counter-led'
    else:
        return 'Symmetric'

# ── London-only typology (same as EDA 3) ──────────────────────
for yr in ['11', '21']:
    cds_p25 = df[f'Cross_Decile_Share_{yr}'].quantile(0.25)
    df[f'Typology_{yr}'] = [
        assign_typology(dom, cds, cds_threshold=cds_p25)
        for dom, cds in zip(df[f'Cascade_Dominance_{yr}'], 
                            df[f'Cross_Decile_Share_{yr}'])
    ]

# ── National-frame typology ───────────────────────────────────
for yr in ['11', '21']:
    cds_p25_nat = df[f'Cross_Decile_Share_nat_{yr}'].quantile(0.25)
    df[f'Typology_nat_{yr}'] = [
        assign_typology(dom, cds, cds_threshold=cds_p25_nat)
        for dom, cds in zip(df[f'Cascade_Dominance_nat_{yr}'], 
                            df[f'Cross_Decile_Share_nat_{yr}'])
    ]
    print(f'Cross_Decile_Share P25 (20{yr}): London={df[f"Cross_Decile_Share_{yr}"].quantile(0.25):.4f}, '
          f'National={cds_p25_nat:.4f}')

print('\n=== Typology Counts ===')
order = ['Cascade-led', 'Symmetric', 'Counter-led', 'Lateral']
for yr in ['11', '21']:
    print(f'\n── 20{yr} ──')
    print(f'{"Type":>14s} {"London":>8s} {"National":>10s} {"Change":>8s}')
    for t in order:
        n_ldn = (df[f'Typology_{yr}'] == t).sum()
        n_nat = (df[f'Typology_nat_{yr}'] == t).sum()
        print(f'  {t:>12s}  {n_ldn:>6d}  {n_nat:>8d}  {n_nat - n_ldn:>+6d}')

## 3b. Typology Cross-Tabulation: Which MSOAs Switch?

In [ ]:
for yr in ['11', '21']:
    print(f'\n{"=" * 60}')
    print(f'TYPOLOGY CROSS-TAB: 20{yr} (rows=London, cols=National)')
    print(f'{"=" * 60}')
    ct = pd.crosstab(df[f'Typology_{yr}'], df[f'Typology_nat_{yr}'],
                     margins=True, margins_name='Total')
    ct = ct.reindex(index=order + ['Total'], columns=order + ['Total'], fill_value=0)
    print(ct.to_string())
    
    same  = (df[f'Typology_{yr}'] == df[f'Typology_nat_{yr}']).sum()
    diff  = len(df) - same
    print(f'\n  Unchanged: {same} ({same/len(df)*100:.1f}%)')
    print(f'  Changed:   {diff} ({diff/len(df)*100:.1f}%)')

## 3c. Who Switches? Profile by Decile and Borough

### 3c-i. Decomposing the Confound: Decile Frame vs External Flows

To understand how much of the typology switching is driven by decile re-framing versus external flows, we check whether MSOAs that changed national decile are more likely to switch typology.

In [ ]:
# ── Confound decomposition: decile-frame vs external-flow effect ──
yr = '21'
df['Same_Decile'] = df['Wealth_Decile'] == df['Wealth_Decile_National']
switched = df[f'Typology_{yr}'] != df[f'Typology_nat_{yr}']

same_dec = df['Same_Decile']
diff_dec = ~df['Same_Decile']

print('=== Confound Decomposition (2021): Decile Frame vs External Flows ===')
print(f'\n  MSOAs that KEPT the same decile number in both frames:')
n_same = same_dec.sum()
n_sw_same = (same_dec & switched).sum()
print(f'    N = {n_same} ({n_same/len(df)*100:.1f}% of all MSOAs)')
print(f'    Typology switched: {n_sw_same} ({n_sw_same/n_same*100:.1f}% of this group)')
print(f'    → These switches are driven ONLY by external-flow effects')
print(f'       (since their decile frame didn\'t change)')

print(f'\n  MSOAs that CHANGED decile between frames:')
n_diff = diff_dec.sum()
n_sw_diff = (diff_dec & switched).sum()
print(f'    N = {n_diff} ({n_diff/len(df)*100:.1f}% of all MSOAs)')
print(f'    Typology switched: {n_sw_diff} ({n_sw_diff/n_diff*100:.1f}% of this group)')
print(f'    → These switches reflect BOTH decile re-framing AND external flows')

print(f'\n  Comparison:')
print(f'    Switch rate if decile unchanged:  {n_sw_same/n_same*100:.1f}%')
print(f'    Switch rate if decile changed:    {n_sw_diff/n_diff*100:.1f}%')
print(f'    ⟹ Decile re-framing approximately {"doubles" if n_sw_diff/n_diff > 1.5*(n_sw_same/n_same) else "increases"} '
      f'the switch rate')
print(f'\n  Attribution (rough):')
print(f'    Switches from external-flow effect alone:  ~{n_sw_same}')
print(f'    Additional switches from decile re-framing: ~{n_sw_diff - int(n_diff * n_sw_same/n_same)}')
print(f'    (Estimated by applying the same-decile switch rate to the changed-decile group)')

In [ ]:
# ── Switchers by decile ───────────────────────────────────────
for yr in ['21']:  # Focus on 2021 (the key period)
    switched = df[f'Typology_{yr}'] != df[f'Typology_nat_{yr}']
    df[f'Switched_{yr}'] = switched
    
    print(f'\n=== 20{yr} Switchers by National Decile ===')
    print(f'{"Decile":>8s} {"N_total":>8s} {"N_switched":>10s} {"% switched":>10s}')
    for d in range(1, 11):
        mask = df['Wealth_Decile_National'] == d
        n_total = mask.sum()
        n_sw = (mask & switched).sum()
        pct = n_sw / n_total * 100 if n_total > 0 else 0
        print(f'  D{d:>2d}    {n_total:>6d}    {n_sw:>8d}    {pct:>8.1f}%')
    
    print(f'\n=== 20{yr} Top 10 Boroughs by Switch Count ===')
    borough_switches = (df.loc[switched, 'ladnm']
                        .value_counts().head(10))
    for boro, count in borough_switches.items():
        total_in_boro = (df['ladnm'] == boro).sum()
        print(f'  {boro:>30s}: {count:>3d} / {total_in_boro} '
              f'({count/total_in_boro*100:.0f}%)')

In [ ]:
# ── Transition detail for 2021 ────────────────────────────────
yr = '21'
switched_mask = df[f'Switched_{yr}']

print(f'\n=== 20{yr} Transition Detail ===')
transitions = (df.loc[switched_mask]
               .groupby([f'Typology_{yr}', f'Typology_nat_{yr}'])
               .size().reset_index(name='count')
               .sort_values('count', ascending=False))
transitions.columns = ['From (London)', 'To (National)', 'N']
print(transitions.to_string(index=False))

## 3d. Typology Scatter: London-only vs National (2021)

In [ ]:
TYPOLOGY_COLORS = {
    'Cascade-led':  '#d6604d',
    'Symmetric':    '#66c2a5',
    'Counter-led':  '#8073ac',
    'Lateral':      '#bdbdbd'
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

yr = '21'
configs = [
    (f'Cascade_Dominance_{yr}', f'Cross_Decile_Share_{yr}', 
     f'Typology_{yr}', 'London-only frame'),
    (f'Cascade_Dominance_nat_{yr}', f'Cross_Decile_Share_nat_{yr}', 
     f'Typology_nat_{yr}', 'National frame'),
]

for idx, (x_col, y_col, typ_col, title) in enumerate(configs):
    ax = axes[idx]
    for t in order:
        mask = df[typ_col] == t
        ax.scatter(df.loc[mask, x_col], df.loc[mask, y_col],
                   c=TYPOLOGY_COLORS[t], label=t, alpha=0.5, s=15, edgecolors='none')
    
    ax.axvline(0.50, color='grey', ls=':', lw=0.8, alpha=0.5)
    ax.axvline(0.52, color='black', ls='--', lw=0.8, alpha=0.5)
    ax.axvline(0.48, color='black', ls='--', lw=0.8, alpha=0.5)
    
    cds_p25 = df[y_col].quantile(0.25)
    ax.axhline(cds_p25, color='black', ls='--', lw=0.8, alpha=0.5)
    
    ax.set_xlabel('Cascade Dominance')
    ax.set_ylabel('Cross Decile Share')
    ax.set_title(title)
    ax.legend(fontsize=8, loc='upper right')
    
    # Count annotation
    counts = df[typ_col].value_counts()
    text = '\n'.join([f'{t}: {counts.get(t, 0)}' for t in order])
    ax.text(0.02, 0.98, text, transform=ax.transAxes, fontsize=8,
            va='top', ha='left', family='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle(f'Fig 21: Typology Scatter — London-only vs National Frame (20{yr})', 
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'fig_21_typology_scatter_comparison_{yr}.png', dpi=150)
plt.show()

## 3e. IMD Validation: Does National-Frame Typology Predict IMD Change Better?

In [ ]:
yr = '21'
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, (typ_col, title) in enumerate([
    (f'Typology_{yr}', 'London-only frame'),
    (f'Typology_nat_{yr}', 'National frame'),
]):
    ax = axes[idx]
    data = [df.loc[df[typ_col] == t, 'IMD_Pctile_Change'].dropna().values
            for t in order]
    bp = ax.boxplot(data, positions=range(len(order)), patch_artist=True, widths=0.6)
    for i, box in enumerate(bp['boxes']):
        box.set_facecolor(TYPOLOGY_COLORS[order[i]])
        box.set_alpha(0.6)
    for med in bp['medians']:
        med.set_color('black')
        med.set_linewidth(2)
    
    ax.set_xticklabels(order, fontsize=9, rotation=15)
    ax.set_ylabel('IMD Pctile Change')
    ax.set_title(title)
    
    # Kruskal-Wallis
    valid_data = [g for g in data if len(g) > 0]
    if len(valid_data) > 1:
        kw_stat, kw_p = stats.kruskal(*valid_data)
        ax.annotate(f'H = {kw_stat:.1f}, p = {kw_p:.2e}',
                    xy=(0.98, 0.02), xycoords='axes fraction', ha='right', fontsize=9,
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle(f'Fig 22: IMD Validation — London-only vs National Frame (20{yr})', 
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'fig_22_imd_validation_comparison_{yr}.png', dpi=150)
plt.show()

In [ ]:
# ── Correlation comparison ────────────────────────────────────
yr = '21'
print(f'=== Cascade metric correlations with IMD_Pctile_Change (20{yr}) ===')
print(f'{"Metric":>28s} {"London r":>10s} {"National r":>12s} {"Diff":>8s}')
print('-' * 62)

for m in ['Cascade_Dominance', 'Cross_Decile_Share', 'Net_Cascade', 'CFI_Churn']:
    ldn_col = f'{m}_{yr}'
    nat_col = f'{m}_nat_{yr}'
    if ldn_col in df.columns and nat_col in df.columns:
        r_ldn = df[ldn_col].corr(df['IMD_Pctile_Change'])
        r_nat = df[nat_col].corr(df['IMD_Pctile_Change'])
        print(f'  {m:>26s}  {r_ldn:>+8.4f}    {r_nat:>+8.4f}  {r_nat - r_ldn:>+8.4f}')

---
# Summary

---

In [ ]:
print('=' * 72)
print('EDA 4 — BOUNDARY EFFECTS SUMMARY')
print('=' * 72)

print('''
PART 1 — Where External Flows Land
──────────────────────────────────────────────
• External MSOA assigned D6; 69% of London MSOAs sit below D6 nationally,
  so for most MSOAs, external connections are "wealthier".
• External inflows were stable (188k → 178k), but outflows nearly doubled
  (219k → 354k). London became a much stronger net exporter by 2021.
• External flows inject volume into BOTH cascade and counter directions
  (wealthier inflow = cascade; wealthier outflow = counter).
  The asymmetry between the two injections determines the net effect.
''')

dom_shift_11 = df['Dom_Shift_11'].mean()
dom_shift_21 = df['Dom_Shift_21'].mean()

print(f'''
PART 2 — Cascade-Counter Balance Shift
──────────────────────────────────────────────
• 2011: External flows nudge dominance toward CASCADE by {dom_shift_11:+.4f}.
  Inflow and outflow boosts were roughly balanced.
• 2021: External flows nudge dominance toward COUNTER by {dom_shift_21:+.4f}.
  The outflow surge (pandemic-era "flight from London") fed counter-cascade
  disproportionately, because for D1–D5 MSOAs, leaving for D6 areas is
  "Outflow_Wealthier" — a counter-cascade component.
• The boundary effect reversed direction between census periods.
''')

yr = '21'
same = (df[f'Typology_{yr}'] == df[f'Typology_nat_{yr}']).sum()
diff = len(df) - same

print(f'''
PART 3 — Typology Stability
──────────────────────────────────────────────
• 2021: {same} MSOAs ({same/len(df)*100:.1f}%) retained their typology classification.
        {diff} MSOAs ({diff/len(df)*100:.1f}%) switched.
• The core narrative holds: counter-cascade dominance intensifying between
  censuses is reinforced, not undermined, by boundary effects.
• London-only analysis UNDERSTATES the counter-cascade shift in 2021.
''')